In [11]:
!pip install requests beautifulsoup4

In [12]:
import requests
import re
import json
from collections import defaultdict

In [13]:
class WikiSearchEngine:
    def __init__(self):
        """Initialize the search engine"""
        self.base_url = "https://en.wikipedia.org/w/api.php"
        self.pages = []
        self.word_locations = defaultdict(list)  # word -> [(page_id, frequency), ...]
        self.stop_words = {'a', 'an', 'the', 'in', 'on', 'at', 'to', 'for', 'of', 'with'}
        self.headers = {
            'User-Agent': 'ColabWikiSearchEngine/1.0 (https://colab.research.google.com; user@example.com)'
        }

    def fetch_wiki_pages(self, topic, num_pages=5):
        """Fetch Wikipedia pages for a given topic"""
        search_params = {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": topic,
            "srlimit": num_pages
        }

        try:
            response = requests.get(self.base_url, params=search_params, headers=self.headers)
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)

            try:
                search_data = response.json()
            except json.JSONDecodeError:
                print(f"Error decoding JSON for search results. Response content: {response.text}")
                return False

            if 'query' not in search_data or 'search' not in search_data['query']:
                print(f"Unexpected JSON structure for search results: {search_data}")
                return False

            search_results = search_data['query']['search']

            for result in search_results:
                content_params = {
                    "action": "query",
                    "format": "json",
                    "prop": "extracts|info",
                    "pageids": result['pageid'],
                    "inprop": "url",
                    "explaintext": True
                }

                content_response = requests.get(self.base_url, params=content_params, headers=self.headers)
                content_response.raise_for_status() # Raise HTTPError for bad responses

                try:
                    page_data_json = content_response.json()
                except json.JSONDecodeError:
                    print(f"Error decoding JSON for page content (pageid: {result['pageid']}). Response content: {content_response.text}")
                    continue # Skip this page and try the next one

                if 'query' not in page_data_json or 'pages' not in page_data_json['query'] or str(result['pageid']) not in page_data_json['query']['pages']:
                    print(f"Unexpected JSON structure for page content (pageid: {result['pageid']}): {page_data_json}")
                    continue

                page_data = page_data_json['query']['pages'][str(result['pageid'])]

                self.pages.append({
                    'id': result['pageid'],
                    'title': page_data['title'],
                    'url': page_data.get('fullurl', f"https://en.wikipedia.org/?curid={result['pageid']}"),
                    'content': page_data.get('extract', '')
                })

                print(f"Retrieved: {page_data['title']}")

            return True

        except requests.exceptions.RequestException as e:
            print(f"Request error while fetching pages: {e}")
            return False
        except Exception as e:
            print(f"An unexpected error occurred: {str(e)}")
            return False

    def build_index(self):
        """Build a word frequency index for each page"""
        self.word_locations.clear()

        for page in self.pages:
            words = re.findall(r'\w+', page['content'].lower())

            word_counts = defaultdict(int)

            for word in words:
                if word not in self.stop_words:
                    word_counts[word] += 1

            for word, count in word_counts.items():
                self.word_locations[word].append((page['id'], count))

    def get_context(self, content, query_words, window=80):
        """Return a short text context around the first matching word"""
        content_lower = content.lower()

        for word in query_words:
            index = content_lower.find(word)
            if index != -1:
                start = max(0, index - window)
                end = min(len(content), index + window)
                return content[start:end].replace("\n", " ")

        return ""

    def search(self, query, num_results=5):
        """
        Search pages using AND / OR operators and simple ranking.

        Ranking:
        1. Number of query words found in the page
        2. Total frequency of those words in the page
        """

        query_upper = query.upper()

        if " AND " in query_upper:
            operator = "AND"
        else:
            operator = "OR"

        query_words = [
            word.lower()
            for word in re.findall(r'\w+', query)
            if word.lower() not in self.stop_words
            and word.upper() not in ["AND", "OR"]
        ]

        if not query_words:
            return []

        page_scores = defaultdict(lambda: {'matches': 0, 'total_freq': 0})

        for word in query_words:
            for page_id, freq in self.word_locations.get(word, []):
                page_scores[page_id]['matches'] += 1
                page_scores[page_id]['total_freq'] += freq

        ranked_results = [
            (page_id, scores['matches'], scores['total_freq'])
            for page_id, scores in page_scores.items()
        ]

        if operator == "AND":
            ranked_results = [
                result for result in ranked_results
                if result[1] == len(query_words)
            ]

        ranked_results.sort(key=lambda x: (x[1], x[2]), reverse=True)

        results = []

        for page_id, matches, total_freq in ranked_results[:num_results]:
            page = next(p for p in self.pages if p['id'] == page_id)

            results.append({
                'title': page['title'],
                'url': page['url'],
                'matching_words': matches,
                'total_frequency': total_freq,
                'context': self.get_context(page['content'], query_words)
            })

        return results

In [14]:
engine = WikiSearchEngine()

engine.fetch_wiki_pages("bird", num_pages=5)

engine.build_index()

Retrieved: Bird
Retrieved: Bird (disambiguation)
Retrieved: Bird & Bird
Retrieved: Larry Bird
Retrieved: Bird, Savage & Bird


In [15]:
results = engine.search("bird OR wings", num_results=5)

for result in results:
    print("Title:", result['title'])
    print("URL:", result['url'])
    print("Matching words:", result['matching_words'])
    print("Total frequency:", result['total_frequency'])
    print("Context:", result['context'])
    print("-" * 80)

Title: Bird
URL: https://en.wikipedia.org/wiki/Bird
Matching words: 2
Total frequency: 92
Context: Birds are a group of warm-blooded vertebrate animals constituting the class Aves
--------------------------------------------------------------------------------
Title: Larry Bird
URL: https://en.wikipedia.org/wiki/Larry_Bird
Matching words: 1
Total frequency: 214
Context: Larry Joe Bird (born December 7, 1956) is an American former professional basketball playe
--------------------------------------------------------------------------------
Title: Bird, Savage & Bird
URL: https://en.wikipedia.org/wiki/Bird,_Savage_%26_Bird
Matching words: 1
Total frequency: 75
Context: Bird, Savage & Bird, was a firm of London merchants transacting business with No
--------------------------------------------------------------------------------
Title: Bird (disambiguation)
URL: https://en.wikipedia.org/wiki/Bird_(disambiguation)
Matching words: 1
Total frequency: 71
Context: A bird is a feathered, winged

In [16]:
results = engine.search("bird AND wings", num_results=5)

for result in results:
    print("Title:", result['title'])
    print("URL:", result['url'])
    print("Matching words:", result['matching_words'])
    print("Total frequency:", result['total_frequency'])
    print("Context:", result['context'])
    print("-" * 80)

Title: Bird
URL: https://en.wikipedia.org/wiki/Bird
Matching words: 2
Total frequency: 92
Context: Birds are a group of warm-blooded vertebrate animals constituting the class Aves
--------------------------------------------------------------------------------
